<a href="https://colab.research.google.com/github/dimon-ton/colab_quiz_generator/blob/main/quiz_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-generativeai
!pip install python-docx
!pip install mistralai
!pip install img2pdf
!pip install typhoon-ocr
!pip install PyPDF2
!apt-get install poppler-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 440.5/440.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.3/160.3 kB 12.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.5/106.5 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 44.0 MB/s eta 0:00:00
  Created wheel for img2pdf: filename=img2pdf-0.6.1-py3-none-any.whl size=51001 sha256=fb4742cc15a81ca4fc2f20eacbc001de4a361bd29c9a5a7d1b1fc8a1651f4c78
  Stored in directory: /root/.cache/pip/wheels/a5/05/56/c05447973db749cd2178b8f95e36f007f0af5f5dce2c6197a5
Successfully built img2pdf
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.2 MB/s eta 0:00:00
Reading package lists... Done
Build

In [ ]:
content_type = 'แบบฝึก' # @param ["แบบฝึก", "แผนการจัดการเรียนรู้"]
quiz_type = 'ปลายภาค' # @param ["ปลายภาค", "กลางภาค"]
semester = 'ภาคเรียนที่ 1' # @param ["ภาคเรียนที่ 1", "ภาคเรียนที่ 2"]
edu_year = '2568' # @param {type:"string"}
class_grade = 'ชั้นประถมศึกษาปีที่ 2'  # @param ["ชั้นประถมศึกษาปีที่ 1", "ชั้นประถมศึกษาปีที่ 2", "ชั้นประถมศึกษาปีที่ 3", "ชั้นประถมศึกษาปีที่ 4", "ชั้นประถมศึกษาปีที่ 5", "ชั้นประถมศึกษาปีที่ 6", "ชั้นมัธยมศึกษาปีที่ 1", "ชั้นมัธยมศึกษาปีที่ 2", "ชั้นมัธยมศึกษาปีที่ 3"]
quiz_number = '30' # @param {type:"string", placeholder:"ใส่จำนวนข้อเป็นตัวเลข"}
score = '1' # @param {type:"string", placeholder:"ข้อละกี่คะแนน"}
total_score = int(quiz_number) * int(score)
period = '60' # @param {type:"string", placeholder:"ใส่เวลาเป็นนาที"}
subject_name = 'ภาษาอังกฤษ' #@param {type:"string", placeholder:"ใส่ชื่อวิชา"}
# set random status
random_choice_status = True #@param {type:"boolean"}
image_include = True #@param {type:"boolean"}
content_source = 'OCR' # @param ["OCR", "LLM"]
choice_amount = 3 #@param {type:"number"}

# LLM provider selection (used when content_source = 'LLM' and for quiz generation)
llm_provider = 'OpenRouter' # @param ["OpenRouter", "Claude CLI"]

In [ ]:
from google.colab import drive
from google.colab import files

import os
import shutil


def upload_files_to_drive(target_folder):
  """Uploads image files from the user's computer to a specified folder in Google Drive.

  Args:
    target_folder: The path to the folder in Google Drive where images will be stored.
      This folder should already exist in your Drive.
  """




  # Remove existing files in the target folder
  if os.path.exists(target_folder):
    for filename in os.listdir(target_folder):
      file_path = os.path.join(target_folder, filename)
      try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
          os.unlink(file_path)
        elif os.path.isdir(file_path):
          shutil.rmtree(file_path)
      except Exception as e:
        print('Failed to delete %s. Reason: %s' % (file_path, e))
    print(f"Existing files removed from '{target_folder}'.")



  # Create the target folder if it doesn't exist
  if not os.path.exists(target_folder):
    os.makedirs(target_folder)
    print(f"Folder '{target_folder}' created in Google Drive.")

  # Prompt the user to upload images
  uploaded = files.upload()

  pdf_uploaded = False
  is_pdf_file = False

  # Save uploaded images to the target folder
  for filename, data in uploaded.items():
    # check if the file name ends with pdf

    if filename.endswith('.pdf'):
      is_pdf_file = True

      if not pdf_uploaded:
        with (open(os.path.join(target_folder, filename), 'wb')) as f:
          f.write(data)
        print(f"PDF file '{filename}' uploaded to '{target_folder}.")
        pdf_uploaded = True # check to upload file pdf only one file
        return (is_pdf_file, os.path.join(target_folder, filename))
      else:
        # Skip additional PDF files
        print(f"PDF file '{filename}' skipped. Only one PDF file is allowed.")

    else:

      with open(os.path.join(target_folder, filename), 'wb') as f:
        f.write(data)
      print(f"File '{filename}' uploaded to '{target_folder}'.")

  return (None, None)


def list_files_with_paths(folder_path):
  """Lists all files with their full paths within a specified folder.

  Args:
    folder_path: The path to the folder you want to list files from.

  Returns:
    A list of strings, where each string is the full path to a file
    within the specified folder.
  """
  file_paths = []
  for root, _, files in os.walk(folder_path):
    for file in files:
      file_paths.append(os.path.join(root, file))
  return file_paths


if content_source == 'OCR':
  if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
  else:
    print("Drive is already mounted.")

  target_folder = "/content/drive/MyDrive/โรงเรียนบ้านโพนแท่น/output_images"
  is_pdf_file, file_name = upload_files_to_drive(target_folder)
  file_lists = list_files_with_paths(target_folder)


Drive is already mounted.
Existing files removed from '/content/drive/MyDrive/โรงเรียนบ้านโพนแท่น/output_images'.


Saving IMG_4464.jpeg to IMG_4464.jpeg
Saving IMG_4465.jpeg to IMG_4465.jpeg
Saving IMG_4466.jpeg to IMG_4466.jpeg
Saving IMG_4467.jpeg to IMG_4467.jpeg
Saving IMG_4468.jpeg to IMG_4468.jpeg
Saving IMG_4469.jpeg to IMG_4469.jpeg
Saving IMG_4470.jpeg to IMG_4470.jpeg
Saving IMG_4471.jpeg to IMG_4471.jpeg
Saving IMG_4472.jpeg to IMG_4472.jpeg
Saving IMG_4473.jpeg to IMG_4473.jpeg
Saving IMG_4474.jpeg to IMG_4474.jpeg
Saving IMG_4475.jpeg to IMG_4475.jpeg
Saving IMG_4476.jpeg to IMG_4476.jpeg
Saving IMG_4477.jpeg to IMG_4477.jpeg
Saving IMG_4478.jpeg to IMG_4478.jpeg
Saving IMG_4479.jpeg to IMG_4479.jpeg
Saving IMG_4480.jpeg to IMG_4480.jpeg
Saving IMG_4481.jpeg to IMG_4481.jpeg
Saving IMG_4482.jpeg to IMG_4482.jpeg
Saving IMG_4483.jpeg to IMG_4483.jpeg
Saving IMG_4484.jpeg to IMG_4484.jpeg
Saving IMG_4485.jpeg to IMG_4485.jpeg
Saving IMG_4486.jpeg to IMG_4486.jpeg
Saving IMG_4487.jpeg to IMG_4487.jpeg
Saving IMG_4488.jpeg to IMG_4488.jpeg
Saving IMG_4489.jpeg to IMG_4489.jpeg
Saving IMG_4

In [ ]:
import img2pdf
import os

def images_to_pdf(image_path, output_pdf_path):

    # image_path is the path of image files in a folder
    img_file_name = os.listdir(image_path)
    img_paths = [f'{image_path}/{img}' for img in img_file_name]

    try:
        # Validate image paths
        for image_path in img_paths:
            if not os.path.exists(image_path):
                raise FileNotFoundError(f"The image file {image_path} does not exist.")

        # Convert images to PDF
        with open(output_pdf_path, "wb") as f:
            f.write(img2pdf.convert(img_paths))

        print(f"PDF successfully created at {output_pdf_path}")

    except Exception as e:
        print(f"An error occurred: {e}")


# upload file to specific folder and get path
# convert images to a pdf file
if content_source == 'OCR':
  if not is_pdf_file:
    image_path = target_folder
    output_pdf_path = "output.pdf"
    images_to_pdf(image_path, output_pdf_path)
  else:
    output_pdf_path = file_name
    print(output_pdf_path)

PDF successfully created at output.pdf


In [ ]:
from mistralai import Mistral, ImageURLChunk, TextChunk, DocumentURLChunk
import base64
from pathlib import Path
import json
from typhoon_ocr import ocr_document
import PyPDF2
import os


from google.colab import userdata


def remove_lines_starting_with(text, prefix):
    """
    Remove lines that start with a specific prefix while preserving the format.

    :param text: The input text as a string.
    :param prefix: The prefix to check for (e.g., '!').
    :return: The text with lines starting with the prefix removed.
    """
    # Split the text into lines
    lines = text.splitlines()

    # Filter out lines that start with the prefix
    filtered_lines = [line for line in lines if not line.strip().startswith(prefix)]

    # Join the remaining lines back into a single string
    cleaned_text = "\n".join(filtered_lines)

    return cleaned_text

def get_text_ocr(pdf_file, ocr_engine='mistral', output_txt_file=None):
    """
    Extracts text from a PDF file using either Mistral OCR or Typhoon OCR and optionally saves it to a text file.

    Args:
        pdf_file: The path to the PDF file.
        ocr_engine: The OCR engine to use ('mistral' or 'typhoon'). Defaults to 'mistral'.
        output_txt_file: The path to the output text file to save the extracted text.
                         If None, the text is not saved to a file. Defaults to None.

    Returns:
        The extracted text as a string, or None if an error occurs.
    """

    if ocr_engine == 'mistral':
      # Retrieve Mistral API key
      mistral_api_key = userdata.get('MISTRAL_API')
      client = Mistral(api_key=mistral_api_key)

      pdf_file_path = Path(pdf_file)
      if not pdf_file_path.is_file():
          print(f"Error: PDF file not found at {pdf_file}")
          return None

      try:
          uploaded_file = client.files.upload(
              file={
                  "file_name": pdf_file_path.stem,
                  "content": pdf_file_path.read_bytes(),
              },
              purpose="ocr",
          )

          signed_url = client.files.get_signed_url(file_id=uploaded_file.id, expiry=1)

          pdf_response = client.ocr.process(document=DocumentURLChunk(document_url=signed_url.url), model="mistral-ocr-latest", include_image_base64=True)

          response_dict = json.loads(pdf_response.model_dump_json())
          # json_string = json.dumps(response_dict, indent=4, ensure_ascii=False)
          # print(response_dict) # Keep for debugging if needed

          ocr_content_list = []

          for page in response_dict["pages"]:
            ocr_content_list.append(page["markdown"])

          msg = "\n\n".join(ocr_content_list)

          msg = remove_lines_starting_with(msg,"!")

          # Save to text file if output_txt_file is provided
          if output_txt_file:
              with open(output_txt_file, 'w', encoding='utf-8') as f:
                  f.write(msg)
              print(f"Extracted text saved to '{output_txt_file}'.")

          return msg
      except Exception as e:
          print(f"Error using Mistral OCR: {e}")
          return None

    elif ocr_engine == 'typhoon':
    # Use Typhoon OCR SDK directly
      try:

          # Load API key from Colab userdata into environment
          typhoon_api_key = userdata.get('TYPHOON_OCR_API_KEY')
          os.environ["TYPHOON_OCR_API_KEY"] = typhoon_api_key

          # Count pages in PDF
          with open(pdf_file, "rb") as f:
              reader = PyPDF2.PdfReader(f)
              num_pages = len(reader.pages)

          ocr_content_list = []

          # Process each page
          for page_num in range(1, num_pages + 1):
              markdown = ocr_document(
                  pdf_or_image_path=pdf_file,
                  task_type="default",     # or "structure"
                  page_num=page_num
              )
              print(f"Page {page_num} OCR done.")

              # Clean output (remove lines starting with "!" like in Mistral OCR)
              msg = remove_lines_starting_with(markdown, "!")
              ocr_content_list.append(msg)

          # Join results from all pages
          extracted_text = "\n\n".join(ocr_content_list)

          # Save to text file if output_txt_file is provided
          if output_txt_file:
              with open(output_txt_file, 'w', encoding='utf-8') as f:
                  f.write(extracted_text)
              print(f"Extracted text saved to '{output_txt_file}'.")


          return extracted_text

      except ImportError:
          print("Error: typhoon_ocr or PyPDF2 package not installed. Please install with `pip install typhoon-ocr PyPDF2`")
          return None
      except FileNotFoundError:
          print(f"Error: PDF file not found at {pdf_file}")
          return None
      except Exception as e:
          print(f"An unexpected error occurred with Typhoon OCR: {e}")
          return None


    else:
        print(f"Error: Invalid OCR engine specified: {ocr_engine}. Choose 'mistral' or 'typhoon'.")
        return None

In [ ]:
import subprocess
import requests
from google.colab import userdata

# Initialize msg outside the function
msg = ""

# Define the OCR engine to use (can be 'mistral' or 'typhoon')
ocr_engine_choice = 'typhoon' # @param ["mistral", "typhoon"]


def call_claude_cli(prompt, timeout=180):
    """Send a prompt to Claude Code CLI and return the text response."""
    result = subprocess.run(
        ['claude', '-p', prompt, '--output-format', 'text'],
        capture_output=True,
        text=True,
        timeout=timeout
    )
    if result.returncode != 0:
        print(f"Claude CLI error: {result.stderr}")
        return None
    return result.stdout.strip()


if content_source == 'OCR':
    if 'output_pdf_path' in globals() and output_pdf_path:
        msg = get_text_ocr(output_pdf_path, ocr_engine=ocr_engine_choice, output_txt_file="output.txt")
    else:
        print("Error: No PDF file path available for OCR.")
        msg = ""

elif content_source == 'LLM':
    llm_prompt = f"""
        Generate lesson content on the subject of {subject_name} for {class_grade} students in Thailand who have no prior knowledge of English.
        The content should cover key points that {class_grade} students should learn, using very simple English words.
        Make sure the content is detailed enough to create {quiz_number} easy quiz questions.
    """

    if llm_provider == 'Claude CLI':
        print("Generating content via Claude CLI...")
        msg = call_claude_cli(llm_prompt)
        if msg is None:
            msg = ""
        else:
            # Save to output.txt for downstream cells
            with open("output.txt", "w", encoding="utf-8") as f:
                f.write(msg)
            print("Content generated by Claude CLI.")

    else:  # OpenRouter
        OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
        url = "https://openrouter.ai/api/v1/chat/completions"
        headers = {
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json"
        }
        data = {
            "model": "anthropic/claude-3.7-sonnet",
            "messages": [{"role": "user", "content": llm_prompt}]
        }
        try:
            response = requests.post(url, headers=headers, json=data)
            response.raise_for_status()
            response_json = response.json()
            msg = response_json['choices'][0]['message']['content']
            print("Content generated by OpenRouter LLM.")
        except requests.exceptions.RequestException as e:
            print(f"Error generating content with OpenRouter LLM: {e}")
            msg = ""

print(msg)

In [ ]:
import requests
import ast
import re
from google.colab import userdata


def create_quiz():

    # Read content from output.txt
    try:
        with open("output.txt", "r", encoding="utf-8") as f:
            msg_content = f.read()
    except FileNotFoundError:
        print("Error: output.txt not found. Please run the previous cell to generate the text file.")
        return None

    example_response = """
        questions = [
            {
                'question': 'What is this?',
                'choices': ['It is a bat.', 'It is running.', 'It is cold.', 'It is mine.'],
                'answer': 'It is a bat.'
            },
        ]
    """

    prompt = f"""
    จากไฟล์เป็นเนื้อหาจาก{content_type}สำหรับการเรียน{subject_name} {class_grade} ให้สร้างข้อสอบ {choice_amount} ตัวเลือก
    เพื่อสอบเก็บคะแนนวิชา{subject_name} โดยใช้สอบนักเรียน{class_grade} {'' if content_type == 'แผนการจัดการเรียนรู้' else 'นักเรียนอ่านประโยคภาษาอังกฤษไม่ค่อยได้'}
    ระดับความยากให้อยู่ในระดับง่าย จำนวน {int(quiz_number) + 2} ข้อ ให้ตอบเป็นรูป Python list เท่านั้น โดยไม่มีคำอธิบายอื่นใด
    ให้แสดงเฉลยด้วย โดยให้เพิ่ม key 'answer' ใน dictionary
    {'' if not image_include else "หากจำเป็นต้องใช้รูปภาพ ให้เพิ่ม key 'image' ใน dictionary และระบุพรอมพ์ที่ใช้สร้างรูปภาพ (เช่น คำอธิบายหรือคำสั่งสำหรับ AI เพื่อสร้างรูปภาพ)"}

    ***ตัวอย่าง output***
    {example_response}

    *** ให้ใช้เนื้อหาต่อไปนี้เพื่อใช้สร้างข้อสอบ ***

    {msg_content}

    ***คำสั่งสำคัญ***
    - choices ต้องอยู่ใน python list มีสมาชิกจำนวน {choice_amount} ตัว โดยไม่มีคำอธิบายหรือข้อความเพิ่มเติม
    - choices ต้องถูกจัดให้อยู่ในรูปของ list ตามที่ระบุ
    - อย่ารวมคำอธิบายอื่นใดนอกจากการตอบเป็น list
    - ใน choices จะต้องมีสมาชิกจำนวน {choice_amount} ตัว ไม่ว่าจะเป็นชั้นไหนก็ตาม และห้ามให้สมาชิกจำนวนทั้ง {choice_amount} ตัว ซ้ำกัน
    """

    print(f"prompt: {prompt}")

    # --- Call the selected LLM provider ---
    if llm_provider == 'Claude CLI':
        print("Generating quiz via Claude CLI...")
        questions = call_claude_cli(prompt, timeout=240)
        if questions is None:
            return None

    else:  # OpenRouter
        OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
        url = "https://openrouter.ai/api/v1/chat/completions"
        headers = {
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json"
        }
        data = {
            "model": "openai/gpt-5",
            "messages": [{"role": "user", "content": prompt}]
        }
        try:
            response = requests.post(url, headers=headers, json=data)
            response.raise_for_status()
            response_json = response.json()
            questions = response_json['choices'][0]['message']['content']
            print("Quiz questions generated by OpenRouter LLM:")
        except requests.exceptions.RequestException as e:
            print(f"Error generating quiz questions with OpenRouter LLM: {e}")
            return None

    print(questions)

    # --- Parse the Python list from the response ---
    try:
        start_index = questions.find("[")
        end_index = questions.rfind("]")

        questions_str = questions[start_index:end_index + 1]

        # Remove inline comments
        clean_text = re.sub(r"\s*#.*$", "", questions_str, flags=re.MULTILINE)
        questions_str = "\n".join(line for line in clean_text.splitlines() if line.strip())

        print(50 * '-')
        print(questions_str.strip())
        print(50 * '-')
        print(start_index, end_index)

        questions_list = ast.literal_eval(questions_str.strip())
        print(questions_list)
    except (SyntaxError, ValueError, AttributeError) as e:
        print(f"Error converting to list: {e}")
        return None

    return questions_list

In [ ]:
question_list = create_quiz()

if question_list is not None:
  if len(question_list) < int(quiz_number):
    print("กรุณาเพิ่มปริมาณเนื้อหาหรือลดจำนวนข้อลง")

prompt: 
        จากไฟล์เป็นเนื้อหาจากแบบฝึกสำหรับการเรียนภาษาอังกฤษ ชั้นประถมศึกษาปีที่ 2 ให้สร้างข้อสอบ 3 ตัวเลือก
        เพื่อสอบเก็บคะแนนวิชาภาษาอังกฤษ โดยใช้สอบนักเรียนชั้นประถมศึกษาปีที่ 2 นักเรียนอ่านประโยคภาษาอังกฤษไม่ค่อยได้
        ระดับความยากให้อยู่ในระดับง่าย จำนวน 32 ข้อ ให้ตอบเป็นรูป Python list เท่านั้น โดยไม่มีคำอธิบายอื่นใด
        ให้แสดงเฉลยด้วย โดยให้เพิ่ม key 'answer' ใน dictionary
        หากจำเป็นต้องใช้รูปภาพ ให้เพิ่ม key 'image' ใน dictionary และระบุพรอมพ์ที่ใช้สร้างรูปภาพ (เช่น คำอธิบายหรือคำสั่งสำหรับ AI เพื่อสร้างรูปภาพ)

        ***ตัวอย่าง output***
        
            questions = [
                {
                    'question': 'What is this?',
                    'choices': ['It is a bat.', 'It is running.', 'It is cold.', 'It is mine.'],
                    'answer': 'It is a bat.'
                },
            ]
        

        *** ให้ใช้เนื้อหาต่อไปนี้เพื่อใช้สร้างข้อสอบ ***

        # Exercise 4

## Example
- A : What is this?
- B : It is a 

In [ ]:
def get_subject_code(level_full: str, subject: str, term: int = 1):
    """
    รับระดับชั้นแบบคำเต็ม เช่น 'ชั้นประถมศึกษาปีที่ 1' หรือ 'ชั้นมัธยมศึกษาปีที่ 3'
    และชื่อรายวิชา พร้อมภาคเรียน (1 หรือ 2) คืนรหัสวิชา

    Parameters:
        level_full (str): ระดับชั้นคำเต็ม เช่น "ชั้นประถมศึกษาปีที่ 1"
        subject (str): ชื่อวิชา เช่น "ภาษาไทย"
        term (int): ภาคเรียน 1 หรือ 2 (default=1)

    Returns:
        str: รหัสวิชา หรือข้อความแจ้งข้อผิดพลาด
    """


      # แปลงคำเต็มเป็นรหัสย่อ
    level_map = {
        "ชั้นประถมศึกษาปีที่ 1": "ป.1",
        "ชั้นประถมศึกษาปีที่ 2": "ป.2",
        "ชั้นประถมศึกษาปีที่ 3": "ป.3",
        "ชั้นประถมศึกษาปีที่ 4": "ป.4",
        "ชั้นประถมศึกษาปีที่ 5": "ป.5",
        "ชั้นประถมศึกษาปีที่ 6": "ป.6",
        "ชั้นมัธยมศึกษาปีที่ 1": "ม.1",
        "ชั้นมัธยมศึกษาปีที่ 2": "ม.2",
        "ชั้นมัธยมศึกษาปีที่ 3": "ม.3"
    }


      # แปลงเป็นรหัสย่อ
    level = level_map.get(level_full)
    if not level:
        return f"ไม่พบข้อมูลระดับชั้น: {level_full}"

    # Mapping structure: { level: { subject: [term1_code, term2_code] } }
    subject_codes = {
        "ป.1": {
            "ภาษาไทย": ["ท11101"],
            "คณิตศาสตร์": ["ค11101"],
            "วิทยาศาสตร์": ["ว11101"],
            "สังคมศึกษา": ["ส11101"],
            "สุขศึกษา": ["พ11101"],
            "พลศึกษา": ["พ11103"],
            "ศิลปะ": ["ศ11101"],
            "การงานอาชีพและเทคโนโลยี": ["ง11101"],
            "ภาษาอังกฤษ": ["อ11101"],
        },
        "ป.2": {
            "ภาษาไทย": ["ท12101"],
            "คณิตศาสตร์": ["ค12101"],
            "วิทยาศาสตร์": ["ว12101"],
            "สังคมศึกษา": ["ส12101"],
            "สุขศึกษา": ["พ12101"],
            "พลศึกษา": ["พ12103"],
            "ศิลปะ": ["ศ12101"],
            "การงานอาชีพและเทคโนโลยี": ["ง12101"],
            "ภาษาอังกฤษ": ["อ12101"],
        },
        "ป.3": {
            "ภาษาไทย": ["ท13101"],
            "คณิตศาสตร์": ["ค13101"],
            "วิทยาศาสตร์": ["ว13101"],
            "สังคมศึกษา": ["ส13101"],
            "สุขศึกษา": ["พ13101"],
            "พลศึกษา": ["พ13103"],
            "ศิลปะ": ["ศ13101"],
            "การงานอาชีพและเทคโนโลยี": ["ง13101"],
            "ภาษาอังกฤษ": ["อ13101"],
        },
        "ป.4": {
            "ภาษาไทย": ["ท14101"],
            "คณิตศาสตร์": ["ค14101"],
            "วิทยาศาสตร์": ["ว14101"],
            "สังคมศึกษา": ["ส14101"],
            "สุขศึกษา": ["พ14101"],
            "พลศึกษา": ["พ14103"],
            "ศิลปะ": ["ศ14101"],
            "การงานอาชีพและเทคโนโลยี": ["ง14101"],
            "ภาษาอังกฤษ": ["อ14101"],
        },
        "ป.5": {
            "ภาษาไทย": ["ท15101"],
            "คณิตศาสตร์": ["ค15101"],
            "วิทยาศาสตร์": ["ว15101"],
            "สังคมศึกษา": ["ส15101"],
            "สุขศึกษา": ["พ15101"],
            "พลศึกษา": ["พ15103"],
            "ศิลปะ": ["ศ15101"],
            "การงานอาชีพและเทคโนโลยี": ["ง15101"],
            "ภาษาอังกฤษ": ["อ15101"],
        },
        "ป.6": {
            "ภาษาไทย": ["ท16101"],
            "คณิตศาสตร์": ["ค16101"],
            "วิทยาศาสตร์": ["ว16101"],
            "สังคมศึกษา": ["ส16101"],
            "สุขศึกษา": ["พ16101"],
            "พลศึกษา": ["พ16103"],
            "ศิลปะ": ["ศ16101"],
            "การงานอาชีพและเทคโนโลยี": ["ง16101"],
            "ภาษาอังกฤษ": ["อ16101"],
        },
        "ม.1": {
            "ภาษาไทย": ["ท21101", "ท21102"],
            "คณิตศาสตร์": ["ค21101", "ค21102"],
            "วิทยาศาสตร์": ["ว21101", "ว21102"],
            "สังคมศึกษา": ["ส21101", "ส21103"],
            "ประวัติศาสตร์": ["ส21102", "ส21104"],
            "สุขศึกษา": ["พ21101", "พ21103"],
            "พลศึกษา": ["พ21102", "พ21104"],
            "ศิลปะ": ["ศ21101", "ศ21102"],
            "การงานอาชีพและเทคโนโลยี": ["ง21101", "ง21102"],
            "ภาษาอังกฤษ": ["อ21101", "อ21102"]
        },
        "ม.2": {
            "ภาษาไทย": ["ท22101", "ท22102"],
            "คณิตศาสตร์": ["ค22101", "ค22102"],
            "วิทยาศาสตร์": ["ว22101", "ว22102"],
            "สังคมศึกษา": ["ส22101", "ส22103"],
            "ประวัติศาสตร์": ["ส22102", "ส22104"],
            "สุขศึกษา": ["พ22101", "พ22103"],
            "พลศึกษา": ["พ22102", "พ22104"],
            "ศิลปะ": ["ศ22101", "ศ22102"],
            "การงานอาชีพและเทคโนโลยี": ["ง22101", "ง22102"],
            "ภาษาอังกฤษ": ["อ22101", "อ22102"]
        },
        "ม.3": {
            "ภาษาไทย": ["ท23101", "ท23102"],
            "คณิตศาสตร์": ["ค23101", "ค23102"],
            "วิทยาศาสตร์": ["ว23101", "ว23102"],
            "สังคมศึกษา": ["ส23101", "ส23103"],
            "ประวัติศาสตร์": ["ส23102", "ส23104"],
            "สุขศึกษา": ["พ23101", "พ23103"],
            "พลศึกษา": ["พ23102", "พ23104"],
            "ศิลปะ": ["ศ23101", "ศ23102"],
            "การงานอาชีพและเทคโนโลยี": ["ง23101", "ง23102"],
            "ภาษาอังกฤษ": ["อ23101", "อ23102"]
        }
    }

    # Validate level and subject
    if level not in subject_codes:
        return f"ไม่พบข้อมูลระดับชั้น: {level_full}"
    if subject not in subject_codes[level]:
        return f"ไม่พบข้อมูลวิชา: {subject} ในระดับ {level_full}"

    # For primary level, always return the only code regardless of term
    if level.startswith("ป."):
        return subject_codes[level][subject][0]

    # For secondary, term must be 1 or 2
    if term not in [1, 2]:
        return f"ภาคเรียนต้องเป็น 1 หรือ 2 เท่านั้น"

    return subject_codes[level][subject][term - 1]

In [ ]:
from docx import Document
from docx.shared import Pt, Inches, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_TAB_ALIGNMENT
from docx.enum.section import WD_SECTION

from docx.oxml import OxmlElement
from docx.oxml.ns import qn
import random


# creat header of quiz

!rm *.docx


def insert_picture(doc, image_path, width_cm=None):
    # Create a paragraph and set alignment to center
    paragraph = doc.add_paragraph()

    # Add the picture to the run within the paragraph
    run = paragraph.add_run()

    if width_cm:
        run.add_picture(image_path, width=Cm(width_cm))
    else:
        run.add_picture(image_path)

    # Set the paragraph alignment to center
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER

def add_border(paragraph):
    # Access the paragraph properties
    p = paragraph._element

    # Create border XML element
    pPr = p.get_or_add_pPr()
    borders = OxmlElement('w:pBdr')

    # Define bottom border
    bottom = OxmlElement('w:bottom')
    bottom.set(qn('w:val'), 'single')  # Type of border (single, double, etc.)
    bottom.set(qn('w:sz'), '4')        # Size of the border (in eights of a point)
    bottom.set(qn('w:space'), '1')     # Space between border and text
    bottom.set(qn('w:color'), '000000') # Border color (hex code)

    # Append the bottom border to the borders element
    borders.append(bottom)
    pPr.append(borders)


def add_line(doc, text1, text2, last_row_status=False):
  # Create a paragraph
  paragraph = doc.add_paragraph()

  # Set the paragraph's tab stops (in this case at 3 inches)
  tab_stops = paragraph.paragraph_format.tab_stops
  tab_stop = tab_stops.add_tab_stop(Cm(16.2), WD_TAB_ALIGNMENT.RIGHT)

  # Add text before the tab

  # line 1
  run = paragraph.add_run(text1)
  run.font.name = 'TH Sarabun New'
  run.font.size = Pt(16)

  run.add_tab()  # Insert a tab character

  run = paragraph.add_run(text2)
  run.font.name = 'TH Sarabun New'
  run.font.size = Pt(16)


  if last_row_status: # if the status is last run this lines of code
    add_border(paragraph)
    doc.add_section(WD_SECTION.CONTINUOUS)

    # set property of section
    section = doc.sections[-1]
    sectPr = section._sectPr
    cols = sectPr.xpath('./w:cols')[0]
    cols.set(qn('w:num'),'2')
    cols.set(qn('w:sep'), '1') # set the line between column


# Create a document
doc = Document()


# Check if 'logo.png' already exists
if not os.path.exists('logo.png'):
  # upload logo pic
  uploaded = files.upload()

  if len(uploaded) > 1:
    print("Please upload only one image file.")
  else:
    # Process the uploaded image here
    for filename, data in uploaded.items():
      # Save or process the image data as needed
      with open(filename, 'wb') as f:
        f.write(data)
      print(f"Image '{filename}' uploaded successfully.")

      # Rename the uploaded file to 'logo.png'
      os.rename(filename, 'logo.png')
      print(f"Image '{filename}' renamed to 'logo.png' successfully.")
else:
  print("logo.png already exists, skipping upload.")



insert_picture(doc, 'logo.png', 2)


# Set the paper size to A4
section = doc.sections[0]
section.page_width = Inches(8.27)  # 210 mm in inches
section.page_height = Inches(11.69)  # 297 mm in inches
section.left_margin = Cm(2.3) # set margin
section.right_margin = Cm(2.3)


subject_code = get_subject_code(
    class_grade,
    subject_name,
    int(semester.split()[1].strip())
)


add_line(doc, f'ข้อสอบวัดผล{quiz_type}', f'{semester}  ปีการศึกษา {edu_year}')
add_line(doc, f'รายวิชา {subject_code} {subject_name}', f'{class_grade}')
add_line(doc, f'จำนวน {quiz_number} ข้อ {total_score} คะแนน เวลา {period} นาที', 'โรงเรียนบ้านโพนแท่น อำเภอเกษตรวิสัย จังหวัดร้อยเอ็ด', True)

# insert explanation heading
explanation_para = doc.add_paragraph()
explanation_run = explanation_para.add_run("คำชี้แจง จงเลือกคำตอบที่ถูกต้องที่สุด")
explanation_run.font.name = "TH Sarabun New"
explanation_run.font.size = Pt(16)

question_list = question_list[:int(quiz_number)] # the number of quiz

answer_correct_list = []
# Loop through questions and add them to the document
for idx, q in enumerate(question_list):
    # Add question number and text
    question_para = doc.add_paragraph()
    question_run = question_para.add_run(f"{idx + 1}. {q['question']}")
    question_run.font.name = "TH Sarabun New"
    question_run.font.size = Pt(16)

    # insert prompt for generating image
    if 'image' in q:
      prompt_para = doc.add_paragraph()
      prompt_run = prompt_para.add_run(f"prompt: {q['image']}")
      prompt_run.font.name = "TH Sarabun New"
      prompt_run.font.size = Pt(10)


      # Shuffle the choices before adding them
    if random_choice_status == True:
      choices = q['choices']
      random.shuffle(choices)
    else:
      choices = q['choices']

    # Add answer choices
    for choice_idx, choice in enumerate(choices, start=97):  # ASCII 'a' = 97
        choice_para = doc.add_paragraph()
        choice_run = choice_para.add_run(f"   {chr(choice_idx)}) {choice}")
        choice_run.font.name = "TH Sarabun New"
        choice_run.font.size = Pt(16)

        print(f"   {chr(choice_idx)}) {choice}")

        # check if the correct choice compared with correct anwser
        if choice == q['answer']:
            answer_correct_list.append(chr(choice_idx))

        # Set paragraph spacing after each choice
        choice_para.paragraph_format.space_after = Pt(3)  # Set space after paragraph


# check if answer_correct_list equals to question_list
if len(answer_correct_list) != len(question_list):
  raise ValueError("answer_correct_list and question_list must have the same length.")


# insert ending paragraph
ending_para = doc.add_paragraph()
ending_run = ending_para.add_run("ขอให้นักเรียนทุกคน โชคดีในการทำข้อสอบน่ะครับ ^_^")
ending_run.font.name = "TH Sarabun New"
ending_run.font.size = Pt(16)
ending_para.paragraph_format.space_before = Pt(50)  # Set space before paragraph
ending_para.alignment = WD_ALIGN_PARAGRAPH.CENTER  # Center-align the paragraph

# insert answer key
doc.add_page_break()
answer_heading_para = doc.add_paragraph()
answer_heading_para_run = answer_heading_para.add_run("เฉลย")
answer_heading_para_run.bold = True
answer_heading_para_run.font.size  = Pt(16)


for idx, (q, a) in enumerate(zip(question_list, answer_correct_list)):
  answer_para = doc.add_paragraph()
  question_run = answer_para.add_run(f"{idx + 1}. {q['answer']} ({a})")
  question_run.font.name = "TH Sarabun New"
  question_run.font.size = Pt(16)


# Save the document
doc.save('exam_template.docx')

print("Document created successfully!")


########################################################################################
from google.colab import files

files.download('exam_template.docx')

logo.png already exists, skipping upload.
   a) It is a book.
   b) They are books.
   c) It is books.
   a) It is a clocks.
   b) It is a clock.
   c) They are clocks.
   a) They are maps.
   b) It is a map.
   c) It is maps.
   a) It is tables.
   b) They are tables.
   c) It is a table.
   a) They are cows.
   b) It is cows.
   c) It is a cow.
   a) It is flowers.
   b) It is a flower.
   c) They are flowers.
   a) They are monkeys.
   b) It is a monkey.
   c) It is monkeys.
   a) It is a bird.
   b) It is birds.
   c) They are birds.
   a) It is a pencil.
   b) They are pencils.
   c) It is pencils.
   a) They are doors.
   b) It is a door.
   c) It is doors.
   a) They are blackboards.
   b) It is a blackboard.
   c) It is blackboards.
   a) It is a rubber.
   b) It is rubbers.
   c) They are rubbers.
   a) They are brooms.
   b) It is a broom.
   c) It is brooms.
   a) It is chairs.
   b) It is a chair.
   c) They are chairs.
   a) It is desks.
   b) They are desks.
   c) It is a

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(len(answer_correct_list))
print(len(question_list))

40
40


# Task
The task is to confirm that the quiz document `exam_template.docx` has been successfully created and downloaded.

## Final Task

### Subtask:
Confirm that the `exam_template.docx` file has been successfully created and downloaded.


## Summary:

### Data Analysis Key Findings
*   The final subtask identified requires confirmation of the successful creation and download of the `exam_template.docx` quiz document.

### Insights or Next Steps
*   The immediate next step is to perform the necessary verification to confirm the presence and successful download of the `exam_template.docx` file.
*   Document the outcome of this confirmation to ensure the task is fully completed.
